In [1]:
#imports
import sys
import numpy as np
import pandas as pd
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#read and create data frames
df1 = pd.read_csv("dataset/comments1.csv")

In [3]:
#get an idea of how the dataframe looks like
print(df1.head())
print(df1.shape)
print(df1.columns)

              kind  commentId  channelId  videoId  authorId  \
0  youtube#comment    1781382      14492    74288   2032536   
1  youtube#comment     289571      14727    79618   3043229   
2  youtube#comment     569077       3314    51826    917006   
3  youtube#comment    2957962       5008    58298   1853470   
4  youtube#comment     673093      21411     1265   2584166   

                                        textOriginal  parentCommentId  \
0  PLEASE LESBIAN FLAG I BEG YOU \n\nYou would ro...              NaN   
1   Apply mashed potato juice and mixed it with curd        3198066.0   
2                         69 missed calls from mars👽              NaN   
3                                               Baaa              NaN   
4    you look like raven from phenomena raven no cap              NaN   

   likeCount                publishedAt                  updatedAt  
0          0  2023-08-15 21:48:52+00:00  2023-08-15 21:48:52+00:00  
1          0  2023-10-02 13:08:22+00:00  202

In [4]:
#compare textOriginal, likeCount, publishedAt
trend1 = df1[["textOriginal", "likeCount", "publishedAt"]]
trend1.isnull().sum()

textOriginal    46
likeCount        0
publishedAt      0
dtype: int64

In [5]:
#clean up the dataframe
trend1 = trend1.dropna(subset=["textOriginal"])
trend1.head()

,textOriginal,likeCount,publishedAt
0,PLEASE LESBIAN FLAG I BEG YOU \n\nYou would ro...,0,2023-08-15 21:48:52+00:00
1,Apply mashed potato juice and mixed it with curd,0,2023-10-02 13:08:22+00:00
2,69 missed calls from mars👽,0,2024-05-31 12:03:12+00:00
3,Baaa,0,2024-02-13 15:48:37+00:00
4,you look like raven from phenomena raven no cap,0,2020-02-15 22:28:44+00:00


In [6]:
trend1['publishedAt'] = pd.to_datetime(trend1['publishedAt'], format='%Y-%m-%d %H:%M:%S%z', utc=True)
oldest = trend1['publishedAt'].min()
newest = trend1['publishedAt'].max()
oldest 
newest  

Timestamp('2025-07-20 15:09:26+0000', tz='UTC')

In [7]:
first_date = trend1["publishedAt"].min()
trend1["days_since_first"] = (trend1["publishedAt"] - first_date).dt.days.astype(float)

In [ ]:
filtered_likes_df1 = trend1[trend1["likeCount"] > 0]
average_like_count1 = filtered_likes_df1["likeCount"].mean()
successful_comments_1 = trend1[trend1["likeCount"] >= average_like_count1]

In [7]:
#do what we did above
all_df = ["comments1.csv","comments2.csv", "comments3.csv", "comments4.csv", "comments5.csv"]
dfs = []  # list to hold each dataframe
for fname in all_df:
    df = pd.read_csv("dataset/" + fname)
    
    trend = df[["textOriginal", "likeCount", "publishedAt"]]
    trend = trend.dropna(subset=["textOriginal"])
    trend['publishedAt'] = pd.to_datetime(trend['publishedAt'], format='%Y-%m-%d %H:%M:%S%z', utc=True)
    first_date = trend["publishedAt"].min()
    trend["days_since_first"] = (trend["publishedAt"] - first_date).dt.days.astype(float)

    filtered_likes_df = trend[trend["likeCount"] > 0]
    average_like_count = filtered_likes_df["likeCount"].mean()
    successful_comments = trend[trend["likeCount"] >= average_like_count]
    dfs.append(successful_comments)

#vertically concat
total_df = pd.concat(dfs, ignore_index=True)

def log1p_mapper(like : float) -> float:
    values = np.log1p(like)
    return values

total_df["likeCountLn"] = total_df['likeCount'].apply(log1p_mapper)



In [3]:
#machine learning imports
from lightgbm import LGBMRegressor   # or LGBMClassifier if you choose a binary label
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

In [8]:
# features and label
X = total_df[["textOriginal", "days_since_first"]] #feature
y = total_df["likeCountLn"]    #label

# text + numeric preprocessing
tfidf = TfidfVectorizer(max_features=4000, ngram_range=(1,2), min_df=5) #https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
#only transform the features for training, they become a matrix
pre = ColumnTransformer([
    ("txt", tfidf, "textOriginal"),
    ("num", "passthrough", ["days_since_first"]),
])

#model receives feature matrix and y_train to create model
model = LGBMRegressor(
    n_estimators=600, #num of trees
    learning_rate=0.05, #slower learning rate per tree but more stable
    subsample=0.9, #each tree takes 90 percent of the training dataset to avoid overfitting
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1
)

pipe = Pipeline([
    ("pre", pre),
    ("model", model),
])

# train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)

# evaluate
y_pred = pipe.predict(X_val)
print("RMSE:", root_mean_squared_error(y_val, y_pred))
print("R^2:", r2_score(y_val, y_pred))

# add predictions back
total_df["pred"] = pipe.predict(X)
total_df["pred_likeCount"] = np.expm1(total_df["pred"])
total_df[["textOriginal","likeCount","pred","pred_likeCount"]].head()

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.201954 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 147489
[LightGBM] [Info] Number of data points in the train set: 34366, number of used features: 3968
[LightGBM] [Info] Start training from score 5.442354
RMSE: 1.3509160136392153
R^2: 0.03518409205199502


,textOriginal,likeCount,pred,pred_likeCount
0,How do you achieve that slick back? 🧐,684,4.991707,146.187530
1,"“If it rains, I’m ruined” was so REALL😭😮😢",88,5.326599,204.736981
2,Can we please go back to Classic beauty?,75,5.368487,213.537998
3,POPULAR..YOURE GONNA BE POPULARR!,701,5.477729,238.302528
4,"Bro my skincare routine is soap, water and a t...",808,6.430255,619.331908
